# 01 — Exemple guidé d'exploration

> ⚠️ **À utiliser comme dépannage, pas comme modèle.**
>
> Ce notebook montre **une** démarche d'exploration possible — pas LA bonne. Il est volontairement basique pour ne pas court-circuiter ta propre démarche. Si tu sais quoi faire, n'ouvre pas ce notebook : invente ta démarche.

L'exemple ci-dessous explore les données sous l'angle d'une **problématique fictive** : *"Quels sont les facteurs qui influencent la rapidité de complétion des tâches sur Lovelace Factory ?"*

Ta problématique sera différente. C'est attendu.

## 1. Préparation

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

pd.set_option('display.max_columns', None)

RAW = Path('..') / 'data' / 'raw'

## 2. Chargement ciblé

Pour cette problématique, je ne charge que ce dont j'ai besoin : `tasks` (pour les durées) et `projects` (pour rattacher à un contexte). Pas besoin de `users`, ni de `comments`, ni de `logs`.

**C'est une décision de minimisation** — démarche RGPD et démarche analytique alignées.

In [ ]:
tasks = pd.read_csv(RAW / 'tasks.csv', parse_dates=['created_at', 'completed_at'])
projects = pd.read_csv(RAW / 'projects.csv')

print(f'{len(tasks)} tâches chargées')
tasks.head(3)

## 3. Inspecter la qualité

Avant tout calcul, je regarde ce qui pourrait fausser l'analyse.

In [ ]:
# Valeurs manquantes par colonne
tasks.isna().sum().sort_values(ascending=False)

In [ ]:
# Outliers sur estimated_hours
print('Min/max estimated_hours :', tasks.estimated_hours.min(), '/', tasks.estimated_hours.max())
print('Quantiles :')
print(tasks.estimated_hours.describe(percentiles=[.01, .25, .5, .75, .99]))

**Observation** : `estimated_hours` contient au moins une valeur à 9999 et probablement des 0 ou négatives. Je vais filtrer.

Note : je documente ici ma décision de filtrage. Dans ton propre notebook, **documente toujours ce que tu enlèves et pourquoi**.

## 4. Construire la variable cible : durée de complétion

In [ ]:
# On garde uniquement les tâches terminées
done = tasks[tasks.status == 'done'].copy()

# Durée de complétion en jours
done['duration_days'] = (done.completed_at - done.created_at).dt.total_seconds() / 86400

# On filtre les valeurs incohérentes
before = len(done)
done = done[
    (done.estimated_hours.between(0.1, 200, inclusive='both'))
    & (done.duration_days > 0)
].copy()
print(f'Filtré : {before - len(done)} tâches retirées (valeurs incohérentes)')
print(f'Il reste {len(done)} tâches terminées exploitables.')

## 5. Premier croisement : priorité vs durée

In [ ]:
# Médiane plutôt que moyenne, plus robuste face aux outliers
by_priority = done.groupby('priority')['duration_days'].agg(['median', 'mean', 'count']).round(2)
by_priority

In [ ]:
# Visualisation
fig = px.box(
    done, x='priority', y='duration_days',
    category_orders={'priority': ['low', 'medium', 'high', 'critical']},
    title='Durée de complétion selon la priorité',
    labels={'duration_days': 'Durée (jours)', 'priority': 'Priorité'},
)
fig.update_layout(showlegend=False)
fig.show()

**Lecture** : à toi de produire ton interprétation. Est-ce que les tâches `critical` sont vraiment traitées plus vite ? Sur quel échantillon ? Quelle est la variance ?

## 6. Et maintenant ?

Cet exemple ne va pas plus loin volontairement. Si tu veux **vraiment** apprendre, c'est à toi de continuer. Quelques pistes :

- Est-ce que la durée varie selon le projet ? L'organisation ? La période de l'année ?
- Y a-t-il un lien entre `estimated_hours` et `duration_days` ? (l'estimation est-elle fiable ?)
- Est-ce que certaines tâches sont *bloquées* longtemps avant d'être complétées ? (croiser avec `activity_logs`)

Mais surtout : **as-tu choisi cette problématique, ou est-ce que tu suis simplement l'exemple ?** Reviens à ta propre question.